### 1. Persiapan dan Memuat Model

Kita memulai tahap akhir ini dengan memuat kembali model dari tahap sebelumnya (atau *base model* yang sudah Anda tentukan). Model tetap dimuat dalam presisi 4-bit untuk efisiensi memori.

In [1]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git@nightly git+https://github.com/unslothai/unsloth-zoo.git

In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

# Memuat model untuk tahap final
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "nxvay/lex02_lora",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


adapter_model.safetensors:   0%|          | 0.00/45.1M [00:00<?, ?B/s]

Unsloth 2026.3.4 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


### 2. Pemasangan LoRA Adapters

Sama seperti sebelumnya, kita menggunakan LoRA agar proses pelatihan dataset regulasi yang berjumlah ribuan ini tidak membebani komputasi secara ekstrem.

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Already have LoRA adapters! We shall skip this step.


### 3. Persiapan Chat Template

Kita kembali memastikan bahwa *chat template* menggunakan standar Llama 3.1, dan membuat fungsi untuk memetakan kolom `messages`.

In [7]:
from unsloth.chat_templates import get_chat_template

# 1. Siapkan Template Chat
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

### 4. Memuat Dataset Utama (Regulasi Daerah)

Inilah inti dari tesis Anda. Kita akan memuat dataset tanya-jawab seputar Peraturan Daerah dan Peraturan Bupati Kutai Kartanegara yang sudah disiapkan di direktori lokal. Terlihat ada **9.000 baris data latih** yang akan digunakan.

In [8]:
from datasets import load_dataset

# ==========================================
# 2. MUAT SEMUA DATASET SEKALIGUS
# ==========================================
data_files = {
    "train": "/content/train.jsonl",
    "validation": "/content/valid.jsonl",
    "test": "/content/test.jsonl"
}

dataset = load_dataset("json", data_files=data_files)

# Terapkan format template chat ke seluruh split
dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['messages', 'text'],
        num_rows: 500
    })
    test: Dataset({
        features: ['messages', 'text'],
        num_rows: 500
    })
})

### 5. Pengecekan Kualitas Konteks Hukum

Mari kita periksa bagaimana dataset hukum ini distrukturkan sebelum masuk ke "otak" AI.

In [11]:
# Cek data ke-0 (Contoh Pertanyaan Retribusi SIUP)
dataset['train'][0]

{'messages': [{'role': 'user',
   'content': 'Berapa biaya retribusi untuk Surat Izin Usaha Perdagangan (SIUP) menurut Peraturan Bupati Kutai Kartanegara Nomor 15 Tahun 2009?\n\nPeraturan Bupati Kutai Kartanegara Nomor 15 Tahun 2009'},
  {'role': 'assistant',
   'content': 'Menurut Peraturan Bupati Kutai Kartanegara Nomor 15 Tahun 2009, biaya retribusi untuk Surat Izin Usaha Perdagangan (SIUP) ditetapkan berdasarkan klasifikasi perusahaan: Rp125.000 untuk perusahaan kecil, Rp250.000 untuk perusahaan menengah, dan Rp500.000 untuk perusahaan besar, berlaku untuk 5 tahun, yang dibebankan kepada pemohon.'}],
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nBerapa biaya retribusi untuk Surat Izin Usaha Perdagangan (SIUP) menurut Peraturan Bupati Kutai Kartanegara Nomor 15 Tahun 2009?\n\nPeraturan Bupati Kutai Kartanegara Nomor 15 Tahun 2009<|eot_id

In [12]:
# Cek data ke-5 (Contoh Pertanyaan Spesifik tentang Pejabat Pengesah)
dataset['train'][5]['text']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSiapa yang menyatakan salinan sesuai dengan aslinya untuk Peraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021?\n\nPeraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nDalam Peraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021, salinan sesuai dengan aslinya dinyatakan oleh Kepala Bagian Hukum Sekretariat Daerah Kabupaten Kutai Kartanegara, Purnomo, SH., MH, dengan NIP 19780605 200212 1 002.<|eot_id|>'

*Teks terlihat sangat rapi dan AI langsung diberikan konteks yang solid, lengkap dengan nama peraturan yang menjadi landasan jawabannya.*

### 6. Konfigurasi Proses Training (SFTTrainer)

Kita siapkan *trainer* dengan parameter yang serupa. Kita akan melatih model selama **3 epoch** dan melakukan evaluasi secara berkala setiap 20 *steps*.

In [13]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,

        # --- PENGATURAN EVALUASI (VALIDASI) ---
        eval_strategy = "steps",
        eval_steps = 20,
        save_strategy = "steps",
        save_steps = 20,

        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/9000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

### 7. Memusatkan Fokus pada Jawaban (Response Masking)

Kita aplikasikan *masking* agar model tidak mencoba menghafal pertanyaan, melainkan fokus pada akurasi jawaban (khususnya akurasi pasal dan nominal denda/pajak).

In [14]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=6):   0%|          | 0/9000 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/9000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

Pembuktian *Masking*:

In [15]:
tokenizer.decode(trainer.train_dataset[5]["input_ids"])

'<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nSiapa yang menyatakan salinan sesuai dengan aslinya untuk Peraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021?\n\nPeraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nDalam Peraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021, salinan sesuai dengan aslinya dinyatakan oleh Kepala Bagian Hukum Sekretariat Daerah Kabupaten Kutai Kartanegara, Purnomo, SH., MH, dengan NIP 19780605 200212 1 002.<|eot_id|>'

In [16]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                              Dalam Peraturan Bupati Kutai Kartanegara Nomor 74 Tahun 2021, salinan sesuai dengan aslinya dinyatakan oleh Kepala Bagian Hukum Sekretariat Daerah Kabupaten Kutai Kartanegara, Purnomo, SH., MH, dengan NIP 19780605 200212 1 002.<|eot_id|>'

### 8. Eksekusi Fine-tuning!

Jalankan proses pelatihan utama. Proses ini mungkin memakan waktu paling lama karena menggunakan 9.000 data.

In [17]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
1.262 GB of memory reserved.


In [18]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,000 | Num Epochs = 3 | Total steps = 3,375
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Step,Training Loss,Validation Loss
20,1.297740,1.206622
40,1.348441,1.142974
60,1.137836,1.111614
80,1.138588,1.087262
100,1.169933,1.065810
120,1.209835,1.050059
140,1.099832,1.029349
160,0.721375,1.020010
180,1.019146,1.011614
200,0.864269,0.998183


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [19]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

7736.0249 seconds used for training.
128.93 minutes used for training.
Peak reserved memory = 4.365 GB.
Peak reserved memory for training = 3.103 GB.
Peak reserved memory % of max memory = 29.973 %.
Peak reserved memory for training % of max memory = 21.307 %.


### 9. Pengujian Akhir (Inference Terhadap Domain Hukum)

Waktunya pembuktian! Kita akan menguji model menggunakan pertanyaan spesifik mengenai peraturan pajak lokal yang sebelumnya selalu gagal dijawab oleh model *base* (sebelum fine-tuning).

In [5]:
# Uji 1: Logika Umum
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Lanjutkan bilangan fibonnaci ini: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 64, use_cache = True,
                         temperature = 0.7, min_p = 0.1)
tokenizer.batch_decode(outputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 26 July 2024\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nLanjutkan bilangan fibonnaci ini: 1, 1, 2, 3, 5, 8,<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nBilangan fibonnaci ini: 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181, 676']

In [6]:
# Uji 2: Pengetahuan Regulasi Lokal (Gunakan Streamer agar terlihat efek mengetik)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Saya punya warung makan kecil di pasar Tenggarong, omset Rp 50 juta/bulan. Berapa persen pajak yang harus dibayar dan pasal mana yang mengatur tarif lengkap beserta cara penghitungannya?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0.75, min_p = 0.1)

Pajak yang harus dibayar adalah 10% (sepuluh persen) dari omset warung makan kecil di pasar Tenggarong.<|eot_id|>


### 10. Menyimpan Model (LoRA, Merged, & GGUF)

Karena ini adalah tahap final, kita akan melakukan tiga jenis penyimpanan yang sangat penting untuk keperluan *deployment*:

**A. Menyimpan LoRA Adapters Saja (Ringan):**

In [22]:
model.save_pretrained("lex03_lora") # Local saving
tokenizer.save_pretrained("lex03_lora")
model.push_to_hub("nxvay/lex03_lora", token = "") # Online saving
tokenizer.push_to_hub("nxvay/lex03_lora", token = "") # Online saving

README.md:   0%|          | 0.00/579 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 23.3kB / 45.1MB            

Saved model to https://huggingface.co/nxvay/lex03_lora


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp0xn26f8c/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

**B. Menggabungkan (Merge) Model ke Standar 16-bit:**

Ini akan menyatukan *base model* dengan *LoRA adapters* menjadi satu model utuh yang mandiri. Ini sangat berguna jika Anda ingin mendeploy model ke VLLM atau platform *cloud inference* lainnya.

In [ ]:
if True: model.save_pretrained_merged("lex03", tokenizer, save_method = "merged_16bit")
if True: model.push_to_hub_merged("nxvay/lex03", tokenizer, save_method = "merged_16bit", token = "")

**C. Konversi ke Format GGUF (Untuk Apple Silicon / Llama.cpp):**

Mengekspor model yang sudah di-*merge* ke dalam format `.gguf` berpresisi tinggi (f16) agar LexIndoLLM siap dijalankan langsung di MacBook Anda menggunakan Ollama.

In [ ]:
if True: model.save_pretrained_gguf("lex03", tokenizer, quantization_method = "f16")
if True: model.push_to_hub_gguf("nxvay/lex03", tokenizer, quantization_method = "f16", token = "")